# 01 · Data preparation
Turn the raw CMMS work orders into one time-to-failure record per pump.

The output, `data/failure_times.csv`, is the input for the reliability modelling in the next notebook. Most of the logic lives in `pm_optimization.py` so it can be reused and tested (see `tests/test_pm_optimization.py`).

`autoreload` picks up edits to `pm_optimization.py` without restarting the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import pm_optimization as pm

## Load the work orders
Each row is one work order from the CMMS export. The order type tells us what kind of job it was:

| Type | Meaning | Used as |
|---|---|---|
| CMRT, CMTA | Corrective maintenance | Failure |
| PMRT, PMTA | Preventive maintenance | Evidence the pump was in service |
| GSRT, EPRT, RWRT | Other work | Ignored |

In [2]:
work_orders = pm.load_work_orders()
print(f"{len(work_orders)} work orders, {work_orders['Equipment'].nunique()} equipment items, "
      f"{work_orders['date'].min():%d %b %Y} to {work_orders['date'].max():%d %b %Y}")
work_orders["Order Type"].value_counts()

711 work orders, 101 equipment items, 01 Apr 2018 to 19 Feb 2022


Order Type
PMRT    552
CMRT    119
CMTA     20
PMTA      9
GSRT      6
EPRT      4
RWRT      1
Name: count, dtype: int64

## Build failure times
`build_failure_times` keeps one record per pump:

- **Failure** (`failed = 1`): hours from the start of the window to the pump's first corrective order. Orders on day 0 are skipped because we don't know how long the pump had been running before the data starts.
- **Suspension** (`failed = 0`): pumps that had PM work but never failed. They were still running at the end, so they are censored at the full window length.

Only 99 of the 101 equipment items end up in the table. The other two have neither a usable failure nor any PM history in the window.

In [3]:
failure_times = pm.build_failure_times(work_orders)

pd.Series({
    "Pumps": len(failure_times),
    "Failures": int(failure_times["failed"].sum()),
    "Suspensions": int((failure_times["failed"] == 0).sum()),
    "Observation window (h)": int(failure_times["hours"].max()),
}).to_frame("value")

,value
Pumps,99
Failures,47
Suspensions,52
Observation window (h),34080


The window is about 34,000 hours, a little under 4 years. Roughly half the pumps failed in that time, so the suspensions carry a lot of information and need to be kept in the fit rather than dropped.

## Save
Write the table out for the next notebook. Pumps are labelled P01 to P99 in time order.

In [4]:
# Label pumps P01, P02, ... instead of their equipment IDs
failure_times = pm.anonymize_units(failure_times)
failure_times.to_csv(pm.FAILURE_TIMES, index=False)
print("Saved", pm.FAILURE_TIMES.relative_to(pm.ROOT))

Saved data\failure_times.csv
